In [50]:
import pandas as pd
import numpy as np

In [51]:
df = pd.read_csv("../data/processed/processed_hospital_data.csv")

print(df.shape)

df.head()

(120000, 33)


,admission_id,patient_id,hospital_id,admit_date,discharge_date,admit_type,ward_type,los_days,readmitted_30d,age,...,diagnosis_categories,cost_category,total_cost_inr,admit_year,admit_month,admit_day,admit_weekday,admit_quarter,discharge_year,discharge_month
0,bcb311d6-0808-498f-8ae1-11abe1fbc08e,6960d399-8398-4db4-9a90-0b329b81bc2b,b22e7b51-2ea6-4611-a8e5-d3d24ec779d8,2024-01-23,2024-01-27,Elective,General,4,0,46,...,Endocrine,Lab,18492,2024,1,23,Tuesday,1,2024,1
1,996ac6db-a192-46a0-b536-503a1c57994b,ec1999d4-f097-4881-91a5-6f9b1f8dac78,8b1e01b7-9f0e-413f-b330-454d271a5c6e,2019-11-18,2019-11-19,OPD,General,1,0,31,...,Respiratory,Lab,1001,2019,11,18,Monday,4,2019,11
2,cc4849fa-b740-487b-beda-3086a18273c0,ed56abac-a0ba-44db-a816-b5da4183b9b5,eae7c9ec-65c8-4d48-92f3-7da7b3e2b227,2019-11-22,2019-11-27,Emergency,General,5,1,53,...,"Neurological,Infectious,Cardiovascular",Room,59696,2019,11,22,Friday,4,2019,11
3,b38cdd03-165c-428c-b221-9f1d4cf8bdeb,90c335d5-dd71-46c8-acd2-656653f421c6,df4431ea-1acf-4e77-963c-6173e9787151,2020-03-05,2020-03-16,Emergency,ICU,11,1,56,...,Cardiovascular,Lab,72588,2020,3,5,Thursday,1,2020,3
4,6a154ec0-92cc-43f5-ada5-ede0a600ce9e,694988e8-9ef5-43e8-aefe-c6b98b1e96fe,8b1e01b7-9f0e-413f-b330-454d271a5c6e,2018-05-30,2018-06-04,Emergency,General,5,1,69,...,"Neoplasm,Respiratory,Gastrointestinal",Lab,9510,2018,5,30,Wednesday,2,2018,6


In [52]:
print("="*60)
print("DATASET INFORMATION")
print("="*60)

print(df.info())

print("\n")

print(df.describe(include="all"))

DATASET INFORMATION
<class 'pandas.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 33 columns):
 #   Column                 Non-Null Count   Dtype
---  ------                 --------------   -----
 0   admission_id           120000 non-null  str  
 1   patient_id             120000 non-null  str  
 2   hospital_id            120000 non-null  str  
 3   admit_date             120000 non-null  str  
 4   discharge_date         120000 non-null  str  
 5   admit_type             120000 non-null  str  
 6   ward_type              120000 non-null  str  
 7   los_days               120000 non-null  int64
 8   readmitted_30d         120000 non-null  int64
 9   age                    120000 non-null  int64
 10  gender                 120000 non-null  str  
 11  patient_state          120000 non-null  str  
 12  insurance_type         86013 non-null   str  
 13  hospital_name          120000 non-null  str  
 14  hospital_state         120000 non-null  str  
 15  hospital

In [53]:
print(df.columns.tolist())

['admission_id', 'patient_id', 'hospital_id', 'admit_date', 'discharge_date', 'admit_type', 'ward_type', 'los_days', 'readmitted_30d', 'age', 'gender', 'patient_state', 'insurance_type', 'hospital_name', 'hospital_state', 'hospital_tier', 'beds', 'teaching', 'primary_icd10', 'primary_diagnosis', 'primary_diag_category', 'total_diagnoses', 'diagnosis_list', 'diagnosis_categories', 'cost_category', 'total_cost_inr', 'admit_year', 'admit_month', 'admit_day', 'admit_weekday', 'admit_quarter', 'discharge_year', 'discharge_month']


In [54]:
print(df.isnull().sum())

df["insurance_type"] = df["insurance_type"].fillna("Unknown")

print("\nMissing Values After Filling\n")

print(df.isnull().sum())

admission_id                 0
patient_id                   0
hospital_id                  0
admit_date                   0
discharge_date               0
admit_type                   0
ward_type                    0
los_days                     0
readmitted_30d               0
age                          0
gender                       0
patient_state                0
insurance_type           33987
hospital_name                0
hospital_state               0
hospital_tier                0
beds                         0
teaching                     0
primary_icd10                0
primary_diagnosis            0
primary_diag_category        0
total_diagnoses              0
diagnosis_list               0
diagnosis_categories         0
cost_category                0
total_cost_inr               0
admit_year                   0
admit_month                  0
admit_day                    0
admit_weekday                0
admit_quarter                0
discharge_year               0
discharg

In [55]:
df["admit_date"] = pd.to_datetime(df["admit_date"])
df["discharge_date"] = pd.to_datetime(df["discharge_date"])

df["admit_year"] = df["admit_date"].dt.year
df["admit_month"] = df["admit_date"].dt.month
df["admit_day"] = df["admit_date"].dt.day
df["admit_weekday"] = df["admit_date"].dt.day_name()
df["admit_quarter"] = df["admit_date"].dt.quarter

In [56]:
df["weekend_admission"] = (
    df["admit_weekday"].isin(
        ["Saturday", "Sunday"]
    )
).astype(int)

In [57]:
df["stay_per_bed"] = (
    df["los_days"] /
    df["beds"]
)

df["cost_per_day"] = (
    df["total_cost_inr"] /
    (df["los_days"] + 1)
)

df["cost_per_bed"] = (
    df["total_cost_inr"] /
    df["beds"]
)

In [58]:
df["num_diagnosis_categories"] = (
    df["diagnosis_categories"]
    .fillna("")
    .str.split(",")
    .apply(lambda x: len([i for i in x if i]))
)

print(df[[
    "diagnosis_categories",
    "num_diagnosis_categories"
]].head())

                     diagnosis_categories  num_diagnosis_categories
0                               Endocrine                         1
1                             Respiratory                         1
2  Neurological,Infectious,Cardiovascular                         3
3                          Cardiovascular                         1
4   Neoplasm,Respiratory,Gastrointestinal                         3


In [59]:
df["diagnosis_density"] = (
    df["total_diagnoses"] /
    (df["los_days"] + 1)
)

df["cost_per_diagnosis"] = (
    df["total_cost_inr"] /
    (df["total_diagnoses"] + 1)
)

df["diagnosis_per_bed"] = (
    df["total_diagnoses"] /
    df["beds"]
)

In [60]:
df["elderly"] = (
    df["age"] >= 65
).astype(int)

df["very_elderly"] = (
    df["age"] >= 80
).astype(int)

df["long_stay"] = (
    df["los_days"] >= 7
).astype(int)

df["very_long_stay"] = (
    df["los_days"] >= 14
).astype(int)

df["multiple_diagnoses"] = (
    df["total_diagnoses"] >= 5
).astype(int)

In [61]:
df["age_diagnosis"] = (
    df["age"] *
    df["total_diagnoses"]
)

In [62]:
df["large_hospital"] = (
    df["beds"] >= df["beds"].median()
).astype(int)

In [63]:
cost_q1 = df["total_cost_inr"].quantile(0.25)

df["low_cost"] = (
    df["total_cost_inr"] <= cost_q1
).astype(int)

In [64]:
cost_q3 = df["total_cost_inr"].quantile(0.75)

df["high_cost"] = (
    df["total_cost_inr"] >= cost_q3
).astype(int)

In [65]:
df["bed_cost_ratio"] = (
    df["beds"] /
    (df["total_cost_inr"] + 1)
)

df["bed_los_ratio"] = (
    df["beds"] /
    (df["los_days"] + 1)
)

In [66]:
def season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    return "Autumn"

In [67]:
df["season"] = df["admit_month"].apply(season)

In [68]:
df["age_los"] = (
    df["age"] *
    df["los_days"]
)

df["age_cost"] = (
    df["age"] *
    df["total_cost_inr"]
)

df["diagnosis_cost"] = (
    df["total_diagnoses"] *
    df["total_cost_inr"]
)

In [69]:
df["teaching_long_stay"] = (
    (df["teaching"] == 1) &
    (df["los_days"] >= 7)
).astype(int)

In [70]:
print(np.isinf(df.select_dtypes(include=np.number)).sum())
df.replace([np.inf, -np.inf], np.nan, inplace=True)

los_days                    0
readmitted_30d              0
age                         0
beds                        0
teaching                    0
total_diagnoses             0
total_cost_inr              0
admit_year                  0
admit_month                 0
admit_day                   0
admit_quarter               0
discharge_year              0
discharge_month             0
weekend_admission           0
stay_per_bed                0
cost_per_day                0
cost_per_bed                0
num_diagnosis_categories    0
diagnosis_density           0
cost_per_diagnosis          0
diagnosis_per_bed           0
elderly                     0
very_elderly                0
long_stay                   0
very_long_stay              0
multiple_diagnoses          0
age_diagnosis               0
large_hospital              0
low_cost                    0
high_cost                   0
bed_cost_ratio              0
bed_los_ratio               0
age_los                     0
age_cost  

,admission_id,patient_id,hospital_id,admit_date,discharge_date,admit_type,ward_type,los_days,readmitted_30d,age,...,large_hospital,low_cost,high_cost,bed_cost_ratio,bed_los_ratio,season,age_los,age_cost,diagnosis_cost,teaching_long_stay
0,bcb311d6-0808-498f-8ae1-11abe1fbc08e,6960d399-8398-4db4-9a90-0b329b81bc2b,b22e7b51-2ea6-4611-a8e5-d3d24ec779d8,2024-01-23,2024-01-27,Elective,General,4,0,46,...,0,0,0,0.019413,71.800000,Winter,184,850632,18492,0
1,996ac6db-a192-46a0-b536-503a1c57994b,ec1999d4-f097-4881-91a5-6f9b1f8dac78,8b1e01b7-9f0e-413f-b330-454d271a5c6e,2019-11-18,2019-11-19,OPD,General,1,0,31,...,0,1,0,0.066866,33.500000,Autumn,31,31031,1001,0
2,cc4849fa-b740-487b-beda-3086a18273c0,ed56abac-a0ba-44db-a816-b5da4183b9b5,eae7c9ec-65c8-4d48-92f3-7da7b3e2b227,2019-11-22,2019-11-27,Emergency,General,5,1,53,...,1,0,0,0.017053,169.666667,Autumn,265,3163888,179088,0
3,b38cdd03-165c-428c-b221-9f1d4cf8bdeb,90c335d5-dd71-46c8-acd2-656653f421c6,df4431ea-1acf-4e77-963c-6173e9787151,2020-03-05,2020-03-16,Emergency,ICU,11,1,56,...,0,0,0,0.002631,15.916667,Spring,616,4064928,72588,0
4,6a154ec0-92cc-43f5-ada5-ede0a600ce9e,694988e8-9ef5-43e8-aefe-c6b98b1e96fe,8b1e01b7-9f0e-413f-b330-454d271a5c6e,2018-05-30,2018-06-04,Emergency,General,5,1,69,...,0,1,0,0.007044,11.166667,Spring,345,656190,28530,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119995,092b2041-97a4-480c-bd0e-6efc2cc25b25,261d105b-6c1d-447e-87e3-5e6eb58d10af,ec4cfe51-9dc6-4c37-aa71-6a806ad081a7,2021-05-29,2021-06-01,Emergency,General,3,0,41,...,1,0,0,0.056390,189.500000,Spring,123,551081,13441,0
119996,36829aa2-4076-43f4-80e3-7e7877536933,802b4e1a-97b9-4c69-9c2b-7dbca1552523,94fcd4c9-f30d-4e64-a627-84d30118ca7c,2024-01-28,2024-02-01,OPD,General,4,0,54,...,1,0,0,0.029737,143.400000,Winter,216,1301940,72330,0
119997,3fdc0376-8c2a-4219-b7bf-5c8c9f7261d4,f4cf1394-2b87-4323-8814-43e3f93f9393,df4431ea-1acf-4e77-963c-6173e9787151,2023-05-30,2023-06-10,Emergency,ICU,11,0,82,...,0,0,0,0.002311,15.916667,Spring,902,6776070,165270,0
119998,32885a57-00af-4cba-94b5-6cbba3829e63,ef232c3e-e6e2-4a4d-a633-579e5ce0bc3d,28879338-cd09-48bb-bedb-730b8ee8ab4e,2023-09-21,2023-09-24,Elective,General,3,1,50,...,0,0,0,0.021871,76.000000,Autumn,150,694950,41697,0


In [71]:
major_categories = [

    "Cardiovascular",
    "Respiratory",
    "Neurological",
    "Endocrine",
    "Infectious",
    "Neoplasm",
    "Gastrointestinal"

]

for category in major_categories:

    column = "has_" + category.lower()

    df[column] = (
    df["diagnosis_categories"]
    .str.contains(
        category,
        case=False,
        na=False
    )
    .astype(int)
)

In [72]:
drop_columns = [

    "admit_date",
    "discharge_date",
    "primary_icd10"

]

df.drop(columns=drop_columns, inplace=True)

print(df.columns)

Index(['admission_id', 'patient_id', 'hospital_id', 'admit_type', 'ward_type',
       'los_days', 'readmitted_30d', 'age', 'gender', 'patient_state',
       'insurance_type', 'hospital_name', 'hospital_state', 'hospital_tier',
       'beds', 'teaching', 'primary_diagnosis', 'primary_diag_category',
       'total_diagnoses', 'diagnosis_list', 'diagnosis_categories',
       'cost_category', 'total_cost_inr', 'admit_year', 'admit_month',
       'admit_day', 'admit_weekday', 'admit_quarter', 'discharge_year',
       'discharge_month', 'weekend_admission', 'stay_per_bed', 'cost_per_day',
       'cost_per_bed', 'num_diagnosis_categories', 'diagnosis_density',
       'cost_per_diagnosis', 'diagnosis_per_bed', 'elderly', 'very_elderly',
       'long_stay', 'very_long_stay', 'multiple_diagnoses', 'age_diagnosis',
       'large_hospital', 'low_cost', 'high_cost', 'bed_cost_ratio',
       'bed_los_ratio', 'season', 'age_los', 'age_cost', 'diagnosis_cost',
       'teaching_long_stay', 'has_cardiov

In [73]:
X = df.drop(columns=["readmitted_30d"])

y = df["readmitted_30d"]

print(X.shape)

print(y.shape)

(120000, 60)
(120000,)


In [74]:
output_path = "../data/processed/feature_engineered_data.csv"

df.to_csv(output_path, index=False)

print(f"Feature engineered dataset saved successfully!")
print(f"Location: {output_path}")
print(f"Shape: {df.shape}")
print("Number of columns:", len(df.columns))

for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")
print("Final Shape:", df.shape)
print("Total Features:", len(df.columns) - 1)

Feature engineered dataset saved successfully!
Location: ../data/processed/feature_engineered_data.csv
Shape: (120000, 61)
Number of columns: 61
1. admission_id
2. patient_id
3. hospital_id
4. admit_type
5. ward_type
6. los_days
7. readmitted_30d
8. age
9. gender
10. patient_state
11. insurance_type
12. hospital_name
13. hospital_state
14. hospital_tier
15. beds
16. teaching
17. primary_diagnosis
18. primary_diag_category
19. total_diagnoses
20. diagnosis_list
21. diagnosis_categories
22. cost_category
23. total_cost_inr
24. admit_year
25. admit_month
26. admit_day
27. admit_weekday
28. admit_quarter
29. discharge_year
30. discharge_month
31. weekend_admission
32. stay_per_bed
33. cost_per_day
34. cost_per_bed
35. num_diagnosis_categories
36. diagnosis_density
37. cost_per_diagnosis
38. diagnosis_per_bed
39. elderly
40. very_elderly
41. long_stay
42. very_long_stay
43. multiple_diagnoses
44. age_diagnosis
45. large_hospital
46. low_cost
47. high_cost
48. bed_cost_ratio
49. bed_los_rati